In [0]:

-- update old records
merge into data.ipldata.customers2_des t
using (select * from data.ipldata.customers2_src 
      where updated_time > (select max(last_run_time) from data.ipldata.cust_control)
        ) as s
on s.customer_id = t.customer_id
and t.is_active = True
when matched 
and (s.phone_number != t.phone_number 
      or s.email != t.email 
      or s.city != t.city
      or s.name != t.name)
then update set
  t.is_active = False,
  t.end_time = s.updated_time;

-- insert updated rows
insert into data.ipldata.customers2_des 
(customer_id, name, phone_number, email, city, start_time, end_time, is_active)
select 
  s.customer_id, s.name, s.phone_number, s.email, s.city, s.updated_time, null, true
from data.ipldata.customers2_src s
join data.ipldata.customers2_des t
on s.customer_id = t.customer_id
where t.is_active = false
and t.end_time = s.updated_time
and (s.phone_number != t.phone_number 
      or s.email != t.email 
      or s.city != t.city
      or s.name != t.name);


-- insert new records
insert into data.ipldata.customers2_des 
(customer_id, name, phone_number, email, city,start_time, end_time, is_active)
select 
  s.customer_id, s.name, s.phone_number, s.email, s.city, s.creation_time, null, true
from data.ipldata.customers2_src s
left join data.ipldata.customers2_des t
on s.customer_id = t.customer_id
and t.is_active = true
where t.customer_id is null
and s.creation_time > (select max(last_run_time) from data.ipldata.cust_control);


-- update control table 
insert into cust_control
(last_run_time, state, insert_records, update_records,error_msg)
select
    current_timestamp,
    'SUCCESS',
    sum(case when start_time >= last_run then 1 else 0 end) as insert_records,
    sum(case when is_active = false and end_time >= last_run then 1 else 0 end) as update_records,
    null
from customers2_des
cross join (
    select max(last_run_time) as last_run
    from cust_control);